# Modeling — Near-Earth Objects (NEO)

Este notebook cobre a fase de **Modeling** do CRISP-DM: treinar e comparar modelos de classificação para prever a variável `hazardous`.

**Ponto de partida:** dados já tratados na fase de Data Preparation (colunas irrelevantes removidas, `est_diameter_min` substituída por `diameter_mean`, `hazardous` convertida para 0/1).

**Estrutura deste notebook:**
1. Carregar e preparar os dados (repete os passos da Data Preparation, para este notebook correr sozinho)
2. Modelo 1 — Regressão Logística
3. Modelo 2 — Random Forest
4. Comparar os modelos
5. Conclusões

## 1. Carregar e preparar os dados

Ajusta o caminho do `read_csv` consoante a localização do `neo.csv` no teu repositório (ex: `../Datasets/neo/raw/neo.csv`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score
)

sns.set(style="whitegrid")

# ajusta este caminho conforme a localização real do ficheiro no teu repositório
df = pd.read_csv("../Datasets/neo/raw/neo.csv")

# --- repete os passos da Data Preparation ---
df = df.drop(columns=['id', 'name', 'orbiting_body', 'sentry_object'])
df['diameter_mean'] = (df['est_diameter_min'] + df['est_diameter_max']) / 2
df = df.drop(columns=['est_diameter_min'])
df['hazardous'] = df['hazardous'].astype(int)

X = df.drop(columns=['hazardous'])
y = df['hazardous']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ['diameter_mean', 'est_diameter_max', 'relative_velocity',
            'miss_distance', 'absolute_magnitude']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print("Treino:", X_train_scaled.shape, "| Teste:", X_test_scaled.shape)

## 2. Modelo 1 — Regressão Logística

Modelo simples e interpretável, bom ponto de partida ("baseline"). Usamos `class_weight='balanced'` porque a classe `hazardous=1` é minoritária (~10%) — isto faz o modelo dar mais importância aos erros nessa classe, em vez de simplesmente prever sempre "não perigoso".

In [ ]:
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_lr, target_names=['Não perigoso', 'Perigoso']))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_lr), 4))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(cm_lr, display_labels=['Não perigoso', 'Perigoso'])
disp.plot(cmap='Blues')
plt.title("Matriz de Confusão — Regressão Logística")
plt.show()

## 3. Modelo 2 — Random Forest

Modelo baseado em várias árvores de decisão, geralmente mais forte com relações não-lineares entre variáveis. Também usamos `class_weight='balanced'` pelo mesmo motivo.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
y_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=['Não perigoso', 'Perigoso']))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_rf), 4))

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(cm_rf, display_labels=['Não perigoso', 'Perigoso'])
disp.plot(cmap='Greens')
plt.title("Matriz de Confusão — Random Forest")
plt.show()

### Quais as variáveis mais importantes para o Random Forest?

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train_scaled.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(7, 4))
importances.plot(kind='barh')
plt.title("Importância das variáveis — Random Forest")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()

## 4. Comparar os modelos

Como a classe `hazardous=1` é minoritária, **accuracy não é uma boa métrica** (um modelo que dissesse sempre "não perigoso" teria ~90% de accuracy e seria inútil). Por isso comparamos com **recall** (não deixar escapar objetos perigosos), **precision**, **F1** e **ROC-AUC**.

Neste problema, um **falso negativo** (dizer que um objeto não é perigoso quando na verdade é) é o erro mais grave — por isso o **recall da classe "Perigoso"** é a métrica mais importante a olhar.

In [ ]:
comparacao = pd.DataFrame({
    'Modelo': ['Regressão Logística', 'Random Forest'],
    'ROC-AUC': [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf)
    ],
    'F1 (Perigoso)': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ]
})
comparacao

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)

plt.figure(figsize=(6, 5))
plt.plot(fpr_lr, tpr_lr, label=f'Regressão Logística (AUC={roc_auc_score(y_test, y_proba_lr):.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={roc_auc_score(y_test, y_proba_rf):.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.xlabel('Falsos Positivos')
plt.ylabel('Verdadeiros Positivos')
plt.title('Curva ROC — Comparação de Modelos')
plt.legend()
plt.show()

## 5. Conclusões (a preencher depois de correr o notebook)

- Qual modelo teve melhor recall na classe "Perigoso"? _(completar)_
- Qual modelo teve melhor ROC-AUC? _(completar)_
- Que variáveis pareceram mais importantes para o Random Forest? _(completar, com base no gráfico de importâncias)_
- Qual modelo escolhem para a fase de Evaluation/entrega final, e porquê? _(completar)_